# 04 · Glucose Management Indicator (GMI) baseline

**Goal:** compare the models from notebooks 01 to 03 with the clinical standard for estimating HbA1c from CGM data, the **Glucose Management Indicator (GMI)**. GMI is shown by every CGM product, so it is the baseline a clinician would expect a model to beat.

**Formula**

GMI (%) = 3.31 + 0.02392 × mean CGM glucose (mg/dL)

**Units.** GMI is an HbA1c estimate in NGSP percent. My target `hba1c` is in IFCC mmol/mol. Every GMI value is therefore converted before any comparison:

mmol/mol = 10.929 × (% − 2.15)

**Protocol.** GMI needs no training, so it is evaluated in two ways:
1. On all 116 patients, to see how well it tracks measured HbA1c.
2. On the same test sets as the earlier notebooks: the saved split, and the five seeds used in `03_models_w_pca_features.ipynb`.

A second variant, **GMI recalibrated**, fits a straight line from GMI to HbA1c on the training data. It is a one-feature model, so it shows whether any gap comes from GMI's formula or from the CGM data itself.

## 1. Setup

In [1]:
# Make the project root importable so `src` can be imported from this notebook
from pathlib import Path
import sys

FILE = Path.cwd().parent.resolve()
sys.path.insert(0, str(FILE))

from src.load_model_data import load_model_data
from src.prepare_splits import load_splits

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [2]:
# All 116 patients with a known HbA1c, with CGM summary features
model_data = load_model_data()
print(f"Patients: {len(model_data)}")
model_data[['patient', 'hba1c', 'cgm_mean']].head()

Patients: 116


,patient,hba1c,cgm_mean
0,1001_0_20210730,115.311,196.829179
1,1002_0_20210504,69.405,136.019620
2,1002_1_20210521,69.405,115.771061
3,1002_2_20210909,72.684,119.082353
4,1003_0_20210831,121.869,151.023898


## 2. What GMI is, and what it is not

GMI is a formula that turns the **average CGM glucose** over the wear period into an HbA1c estimate. It was derived from population data, so it is a population-level estimate. It does not learn anything from the patient's other features.

Three things to keep in mind when reading the comparison:
- **The window is short.** My CGM series cover about 1 to 2 weeks per patient. HbA1c reflects about the previous 2 to 3 months. GMI is usually recommended for at least about two weeks of CGM data, so my patients are at the lower edge of what it was designed for.
- **Measured HbA1c is the reference.** GMI is compared with the lab value, as the models are, so any gap is a gap between two estimates of the same quantity.
- **`cgm_mean` covers every reading** the patient has, so it is the GMI input as defined by the CGM products.

## 3. Step 1: units and the formula

Two conversions are needed:
1. GMI formula: mean glucose (mg/dL) → GMI (%).
2. IFCC conversion: percent → mmol/mol, so GMI can be compared with `hba1c`.

The checks below confirm both conversions on reference values before they are used on the data.

In [3]:
# GMI formula and IFCC conversion
GMI_INTERCEPT = 3.31
GMI_SLOPE = 0.02392
IFCC_OFFSET = 2.15
IFCC_FACTOR = 10.929


def gmi_percent(mean_glucose_mg_dl):
    """GMI in NGSP percent from mean CGM glucose in mg/dL."""
    return GMI_INTERCEPT + GMI_SLOPE * mean_glucose_mg_dl


def percent_to_mmol_mol(hba1c_percent):
    """Convert NGSP percent to IFCC mmol/mol."""
    return IFCC_FACTOR * (hba1c_percent - IFCC_OFFSET)


def gmi_mmol_mol(mean_glucose_mg_dl):
    """GMI expressed in mmol/mol, the unit of the hba1c target."""
    return percent_to_mmol_mol(gmi_percent(mean_glucose_mg_dl))


def metrics(y_true, y_pred, name):
    """Regression metrics plus bias (mean of predicted minus measured, in mmol/mol)."""
    return {
        'model': name,
        'r2': r2_score(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'bias': np.mean(y_pred - y_true),
        'pearson_r': np.corrcoef(y_true, y_pred)[0, 1],
    }


# Check 1: an average glucose of 154 mg/dL should give about 6.9 % (a known reference point)
print(f"Mean glucose 154 mg/dL -> GMI {gmi_percent(154):.2f} %")

# Check 2: 7.0 % should convert to about 53 mmol/mol (the standard IFCC value)
print(f"7.0 % -> {percent_to_mmol_mol(7.0):.1f} mmol/mol")

# Check 3: 53 mmol/mol should convert back to about 7.0 %
print(f"53 mmol/mol -> {53 / IFCC_FACTOR + IFCC_OFFSET:.2f} %")

Mean glucose 154 mg/dL -> GMI 6.99 %
7.0 % -> 53.0 mmol/mol
53 mmol/mol -> 7.00 %


## 4. Step 2: GMI for every patient

GMI is computed from the mean CGM glucose column `cgm_mean` and stored in mmol/mol. The two columns are compared side by side, so a unit error would show up as a large gap.

In [4]:
# GMI in % and in mmol/mol for every patient
model_data['gmi_percent'] = gmi_percent(model_data['cgm_mean'])
model_data['gmi'] = gmi_mmol_mol(model_data['cgm_mean'])

model_data[['cgm_mean', 'gmi_percent', 'gmi', 'hba1c']].describe().round(2)

,cgm_mean,gmi_percent,gmi,hba1c
count,116.00,116.00,116.00,116.00
mean,144.66,6.77,50.49,75.77
std,30.75,0.74,8.04,27.44
min,68.80,4.96,30.66,23.50
25%,119.20,6.16,43.84,54.92
50%,142.76,6.72,50.00,68.31
75%,165.34,7.26,55.90,94.82
max,221.86,8.62,70.68,165.59


## 5. Step 3: GMI against measured HbA1c (all 116 patients)

This is the first look at how well GMI tracks measured HbA1c. It uses no split and no training. Points on the dashed line are perfect agreement. Points above it are overestimates.

In [5]:
# GMI vs measured HbA1c for all patients
all_metrics = pd.DataFrame([metrics(model_data['hba1c'], model_data['gmi'], 'GMI (raw, all patients)')])
print(all_metrics.round(3).to_string(index=False))

fig = go.Figure(go.Scatter(
    x=model_data['hba1c'], y=model_data['gmi'],
    mode='markers', marker=dict(size=7, opacity=0.6), name='patients'
))
lo, hi = model_data['hba1c'].min(), model_data['hba1c'].max()
fig.add_trace(go.Scatter(
    x=[lo, hi], y=[lo, hi], mode='lines',
    line=dict(dash='dash', color='grey'), name='perfect agreement'
))
fig.update_layout(
    title='GMI vs measured HbA1c (mmol/mol)',
    xaxis_title='Measured HbA1c (mmol/mol)',
    yaxis_title='GMI converted to mmol/mol',
    height=500, width=600
)
fig.show()

                  model     r2   rmse    mae    bias  pearson_r
GMI (raw, all patients) -0.683 35.442 26.017 -25.281      0.442


In [6]:
# Save the GMI vs measured HbA1c figure for the README
from pathlib import Path
FIG_DIR = Path.cwd().parent / 'figures'
FIG_DIR.mkdir(exist_ok=True)
fig.write_image(FIG_DIR / 'gmi_vs_hba1c.png', width=600, height=500, scale=2)

**Observations (all 116 patients)**
- Raw GMI moves in the same direction as measured HbA1c (Pearson r = 0.44), but its level is far off. The bias is -25 mmol/mol: on average GMI is about 25 mmol/mol below the measured value (mean GMI 50 against mean HbA1c 76).
- The unit conversion was checked against reference values, so the gap is not a units error. The likely causes are the 1–2 week CGM window, the population, and the fact that HbA1c and CGM were not necessarily measured at the same time. This data cannot separate them.

## 6. Step 4: the saved test split (24 patients)

The saved split is the one used by the notebooks 01 and 02 models. GMI raw needs no training. GMI recalibrated fits a line from GMI to HbA1c on the 92 training patients and is then scored on the 24 test patients.

In [7]:
# Saved split: GMI raw and GMI recalibrated on the same test patients as the earlier models
splits = load_splits()
train_s = splits['train'].copy()
test_s = splits['test'].copy()
for d in (train_s, test_s):
    d['gmi'] = gmi_mmol_mol(d['cgm_mean'])

recal = LinearRegression().fit(train_s[['gmi']], train_s['hba1c'])

split_rows = [
    metrics(test_s['hba1c'], test_s['gmi'], 'GMI (raw)'),
    metrics(test_s['hba1c'], recal.predict(test_s[['gmi']]), 'GMI (recalibrated on train)'),
]
print(pd.DataFrame(split_rows).round(3).to_string(index=False))
print(f"\nRecalibration fitted on train: hba1c = {recal.coef_[0]:.3f} * gmi + {recal.intercept_:.3f}")

                      model     r2   rmse    mae    bias  pearson_r
                  GMI (raw) -0.910 41.116 30.809 -30.577      0.419
GMI (recalibrated on train)  0.115 27.978 23.828  -7.134      0.419

Recalibration fitted on train: hba1c = 1.492 * gmi + -1.038


**Observations (saved split, 24 test patients)**
- Raw GMI scores R² = -0.91 on the test patients, far below the mean-only predictor (-0.04). Its bias of -31 mmol/mol dominates the error.
- After a straight-line recalibration fitted on the training patients, the same single feature reaches R² = 0.12. That is in the range of the linear models (0.07 to 0.12) and below the Random Forest (0.17).
- The fitted slope of 1.49 means that across these patients each point of GMI corresponds to about 1.5 mmol/mol of measured HbA1c. The measured values spread out more than GMI does.

## 7. Step 5: the five seeds from notebook 03

The same five seeds and the same 80/20 splits as `03_models_w_pca_features.ipynb`. This gives mean and standard deviation over five test sets, to compare directly with the linear, Random Forest and mean-only rows in that notebook.

In [8]:
# GMI raw and GMI recalibrated on the five seed splits
SEEDS = [0, 1, 2, 3, 4]
seed_rows = []

for seed in SEEDS:
    train_raw, test_raw = train_test_split(model_data, test_size=0.2, random_state=seed)
    train_raw = train_raw.copy()
    test_raw = test_raw.copy()

    recal_seed = LinearRegression().fit(train_raw[['gmi']], train_raw['hba1c'])
    predictions = {
        'GMI (raw)': test_raw['gmi'],
        'GMI (recalibrated)': recal_seed.predict(test_raw[['gmi']]),
    }
    for name, pred in predictions.items():
        row = metrics(test_raw['hba1c'], pred, name)
        row['seed'] = seed
        seed_rows.append(row)

seed_df = pd.DataFrame(seed_rows)
print(seed_df[['seed', 'model', 'r2', 'rmse', 'mae', 'bias']].round(3).to_string(index=False))
print()
print(seed_df.groupby('model')[['r2', 'rmse', 'mae', 'bias']].agg(['mean', 'std']).round(3))

 seed              model     r2   rmse    mae    bias
    0          GMI (raw) -0.809 42.775 30.936 -30.284
    0 GMI (recalibrated)  0.039 31.172 23.046  -7.157
    1          GMI (raw) -0.357 31.766 23.058 -21.376
    1 GMI (recalibrated)  0.291 22.959 19.319   3.158
    2          GMI (raw) -1.074 42.338 33.768 -33.768
    2 GMI (recalibrated)  0.163 26.891 19.294 -10.688
    3          GMI (raw) -0.568 39.204 30.390 -29.710
    3 GMI (recalibrated)  0.339 25.463 20.816  -5.614
    4          GMI (raw) -0.530 35.215 25.649 -25.196
    4 GMI (recalibrated)  0.306 23.722 17.471   0.559

                       r2           rmse            mae          bias       
                     mean    std    mean    std    mean   std    mean    std
model                                                                       
GMI (raw)          -0.668  0.279  38.260  4.726  28.760  4.32 -28.067  4.825
GMI (recalibrated)  0.228  0.125  26.042  3.250  19.989  2.08  -3.948  5.686


**Observations (five seeds, same splits as notebook 03)**
- Raw GMI: mean R² -0.67 ± 0.28, with a bias of about -28 mmol/mol on every split.
- GMI recalibrated: mean R² 0.23 ± 0.13, with a bias close to zero on average (-3.9 ± 5.7). That is the same as the linear regression on PCA features (0.24 ± 0.13) and better than the Random Forest (0.13 ± 0.14).
- A single CGM number, properly calibrated, does as well as the machine learning models in notebook 03. On this data the extra PCA components and the forest add nothing measurable.

## 8. Summary

Five seeds, the same test sets for every row. Reference rows are copied from `03_models_w_pca_features.ipynb`; bias is not reported there.

| Model | Mean R² | Std R² | Mean RMSE | Mean MAE | Mean bias |
|---|---|---|---|---|---|
| Mean-only predictor (nb 03) | -0.072 | 0.075 | 30.65 | 24.91 | — |
| Linear regression on PCs (nb 03) | 0.235 | 0.131 | 25.90 | 19.86 | — |
| Random Forest, tuned (nb 03) | 0.134 | 0.136 | 27.55 | 21.30 | — |
| GMI (raw) | -0.668 | 0.279 | 38.26 | 28.76 | -28.07 ± 4.83 |
| GMI (recalibrated on train) | 0.228 | 0.125 | 26.04 | 19.99 | -3.95 ± 5.69 |

### Observations
- **Published GMI is biased low for this cohort.** Used as-is, it underestimates HbA1c by about 28 mmol/mol on average, and it is far worse than the mean-only predictor.
- **Once recalibrated, one CGM feature matches the best models.** The GMI-recalibrated line scores the same as the linear model on PCA features and better than the forest. The extra complexity in notebook 03 does not buy anything measurable.

### Why both struggle on 2-week windows
- **Time mismatch.** My CGM covers about 1 to 2 weeks. HbA1c reflects the previous 2 to 3 months. A short window sees only part of what HbA1c integrates.
- **The information is limited, not the algorithm.** A mean glucose over the window explains about 19% of the variance (r² ≈ 0.44² ≈ 0.19 on all patients). No model can extract more from the window than the window contains.
- **Both sides are estimates.** GMI is a population formula, and measured HbA1c is a lab value. Disagreement between them is expected, especially at short windows.

### Next steps
- Test GMI on data with longer CGM windows, or with HbA1c dates close to the CGM period. Either would show whether the gap comes from the window.
- Compare future models against **GMI recalibrated**, not against the mean-only predictor alone. It is the clinically meaningful baseline.